# Tool Calling Fine-Tuning (QLoRA on T4 GPU)
Bu notebook, **Qwen2.5-0.5B** modeli uzerinde Tool / Function Calling yetenegini egitmek icin hazirlanmistir.

**Hedef GPU:** Google Colab T4 (16GB VRAM, fp16) / Kaggle (GPU T4 x2)

---

### 1. GPU Kontrolu

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise SystemError(
        "HATA: GPU runtime bulunamadi!\n"
        "Lutfen Google Colab'de 'T4 GPU' veya Kaggle'da 'GPU T4 x2' secin!"
    )

device_count = torch.cuda.device_count()
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Tespit Edilen GPU Sayisi: {device_count}")
for i in range(device_count):
    vram = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  [GPU {i}] {torch.cuda.get_device_name(i)} - {vram:.1f} GB VRAM")
print(f"bf16 destegi: {torch.cuda.is_bf16_supported()} (T4 icin False olmasi normaldir; float16 kullanilacaktir)")
if device_count > 1:
    print("\n[BILGI] Kaggle 2x T4 ortamindasiniz. Qwen2.5-0.5B modeli ~4-5 GB VRAM gerektirdigi icin")
    print("       egitim CUDA_VISIBLE_DEVICES=0 ile 1. GPU uzerinde kararlı ve en hizli sekilde calistirilacaktir.")


### 2. Projeyi Klonla ve Kur

In [ ]:
import os

REPO_URL = "https://github.com/fatihkadim/tool-calling-ft.git"

# Calisma ortami tespiti: Google Colab (/content) veya Kaggle (/kaggle/working)
if os.path.exists("/content"):
    PROJECT_DIR = "/content/tool-calling-ft"
elif os.path.exists("/kaggle/working"):
    PROJECT_DIR = "/kaggle/working/tool-calling-ft"
else:
    PROJECT_DIR = os.path.abspath(".")

if os.path.exists("/content") or os.path.exists("/kaggle/working"):
    if not os.path.exists(PROJECT_DIR):
        print(f"Repo klonlaniyor: {PROJECT_DIR}")
        !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}
    !git pull origin main
else:
    print("Yerel/Ozel calisma ortami:", os.getcwd())

!pwd


### 3. Bagimliliklarin Kurulumu

In [ ]:
!pip install -q --upgrade pip
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q "transformers>=4.46" "peft>=0.13" "bitsandbytes>=0.44" "datasets>=3.0" "trl>=0.11" "accelerate>=1.0" pyyaml tqdm pandas matplotlib

# uv_build backend'ini kur, sonra projeyi editable olarak yukle
!pip install -q "uv_build>=0.11.7,<0.12.0"
!pip install -q --no-build-isolation -e .

### 4. Veri Setini Hazirla

In [ ]:
!python -m tool_calling_ft.data.prepare_dataset

### 5. T4 Icin QLoRA Config Olustur

> **Onemli:** T4 GPU **bf16 desteklemiyor**, bu yuzden `float16` kullanmamiz gerekiyor.
> Ayrica T4'un 16GB VRAM'i ile `max_seq_len=2048` guvenli calisir.

In [ ]:
import yaml

config = {
    "method": "qlora",
    "base_model": "Qwen/Qwen2.5-0.5B",
    "dataset": "NousResearch/hermes-function-calling-v1",
    "output_dir": "checkpoints/qlora",
    "quantization": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",  # T4 bf16 desteklemiyor!
        "bnb_4bit_use_double_quant": True,
    },
    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    },
    "training": {
        "epochs": 3,
        "batch_size": 4,
        "grad_accum_steps": 2,
        "learning_rate": 2e-4,
        "max_seq_len": 2048,
        "warmup_ratio": 0.05,
        "save_steps": 200,
        "seed": 42,
    },
}

with open("configs/qlora_t4.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("T4 config yazildi: configs/qlora_t4.yaml")
print(yaml.dump(config, default_flow_style=False))

### 6. Egitimi Baslat

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.training.train --config configs/qlora_t4.yaml

### 7. Canli Test (Inference Demo)
Egittiginiz modele ornek bir arac verip ciktisini canli test edin.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

base_model_name = "Qwen/Qwen2.5-0.5B"
adapter_path = "checkpoints/qlora"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

# Ornek test sorusu ve Tool semasi
system_prompt = """You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags.
<tools>
[{"type": "function", "function": {"name": "get_current_weather", "description": "Get current weather for a city", "parameters": {"type": "object", "properties": {"location": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["location"]}}}]
</tools>
For each function call return a json object with function name and arguments within <tool_call> </tool_call> tags."""

user_query = "What is the weather in Tokyo in celsius?"

prompt = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Stop tokens: hem <|im_end|> hem <|endoftext|>
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
stop_ids = [tokenizer.eos_token_id]
if isinstance(im_end_id, int) and im_end_id != tokenizer.eos_token_id:
    stop_ids.append(im_end_id)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, eos_token_id=stop_ids)

response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print("=" * 50)
print("MODEL CIKTISI:")
print(response.strip())
print("=" * 50)

### 8. Degerlendirme

In [ ]:
# QLoRA Modelini Degerlendirme:
!CUDA_VISIBLE_DEVICES=0 python -m tool_calling_ft.eval.harness --method qlora --adapter checkpoints/qlora --dataset data/processed/eval_subset.jsonl

### 9. Sonuclari Yedekle (Google Drive)

In [ ]:
import os

# 1. Secenek: Google Colab Drive Yedekleme
if os.path.exists("/content"):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_dest = "/content/drive/MyDrive/tool_calling_all_results"
        os.makedirs(drive_dest, exist_ok=True)
        !cp -r checkpoints {drive_dest}/ 2>/dev/null || echo 'checkpoints klasoru kopyalanamadi'
        !cp -r reports {drive_dest}/ 2>/dev/null || echo 'reports klasoru kopyalanamadi'
        print(f"Training sonuclari Drive'a kaydedildi -> {drive_dest}")
    except Exception as e:
        print("Google Drive baglantisi atlandi:", e)

# 2. Secenek: ZIP arsivi (Hem Colab hem Kaggle ile %100 uyumlu)
!zip -q -r all_results.zip checkpoints reports 2>/dev/null || true
if os.path.exists("all_results.zip"):
    if os.path.exists("/kaggle/working") and os.getcwd() != "/kaggle/working":
        !cp all_results.zip /kaggle/working/
        print("all_results.zip Kaggle /kaggle/working dizinine kopyalandi!")
        print("Notebook tamamlandiginda sag paneldeki 'Output' sekmesinden dogrudan indirebilirsiniz.")
    else:
        print("\nall_results.zip arsivi olusturuldu! Dosyalar panelinden indirebilirsiniz.")


In [ ]:
# Alternatif: zip indirmek icin
!zip -r tool_calling_results.zip checkpoints reports 2>/dev/null
from google.colab import files

files.download('tool_calling_results.zip')